# FIRA Cup — YOLOv8n Traffic Sign Detection

This notebook is the reproducible training/export record for the project. It reconstructs the workflow from the supplied source files: YOLOv8n training, dataset preparation, ONNX export, and CPU inference.

**Dataset note:** the original self-collected dataset is intentionally not committed. The first experiment used 400+ images; after validating the approach, the dataset was expanded to 800+ images for further fine-tuning.

> This notebook does not fabricate training metrics or benchmark numbers that were not preserved in the source project. Execute the cells with the original dataset to reproduce measurements.

## 1. Environment
Install the training/export dependencies in the repository environment.

In [ ]:
%pip install ultralytics onnxruntime opencv-python

## 2. Imports and project configuration

In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO

PROJECT_ROOT = Path('.').resolve()
DATA_CONFIG = PROJECT_ROOT / 'configs' / 'data.yaml'
# MODEL_DIR = PROJECT_ROOT / 'models'
print(PROJECT_ROOT)
print(DATA_CONFIG)

## 3. Dataset contract
The dataset stays outside Git. Update `configs/data.yaml` so `path` points to the local YOLO dataset root.

In [ ]:
with open(DATA_CONFIG, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg

Expected structure:

```text
dataset/
├── images/
│   ├── train/
│   └── val/
└── labels/
    ├── train/
    └── val/
```

Classes are fixed as `stop`, `right`, `left`, `straight` with IDs 0–3.

## 4. Optional augmentation / preparation
The original project used image-level augmentations including blur, motion blur, brightness, contrast, gamma, noise, HSV shifts, sharpening, and JPEG compression.

The cleaned implementation is available as `tools/augment_dataset.py`.

In [ ]:
!python tools/augment_dataset.py --help

## 5. Train YOLOv8n
The supplied project establishes YOLOv8n as the training architecture. Because the original training-run hyperparameters were not preserved, set the experiment values here to match the run you want to reproduce.

In [ ]:
model = YOLO(str(PROJECT_ROOT / 'assets' / 'yolov8n_fira_best.pt'))

results = model.train(
    data=str(DATA_CONFIG),
    imgsz=416,           # deployment size (previous run used 640)
    epochs=100,
    patience=25,
    batch=16,
    optimizer='AdamW',
    lr0=0.0001,
    seed=0,
    fliplr=0.0,          # never flip: it would swap left/right
    flipud=0.0,
    mosaic=0.0,
    hsv_h=0.0, hsv_s=0.1, hsv_v=0.1,
    degrees=1, translate=0.03, scale=0.1,
    project=str(PROJECT_ROOT / 'runs'),
    name='fira_yolov8n_park',
)

best = YOLO(str(PROJECT_ROOT / 'runs' / 'fira_yolov8n_park' / 'weights' / 'best.pt'))
metrics = best.val(data=str(DATA_CONFIG), imgsz=416)
print(metrics.box.map50)
print(metrics.box.maps)      # per-class mAP: check that "park" is reasonable

best.export(format='onnx', imgsz=416)   # only the size you deploy

## 6. Export to ONNX
The original project contained 416×416 and 640×640 ONNX artifacts. Export both resolutions when reproducing the deployment package.

In [ ]:
best_weights = PROJECT_ROOT / 'runs' / 'fira_yolov8n_park' / 'weights' / 'best.pt'
trained = YOLO(str(best_weights))

export_416 = trained.export(format='onnx', imgsz=416)
# export_640 = trained.export(format='onnx', imgsz=640)
print(export_416)
# print(export_640)

## 7. CPU inference check
Run the exported model through the cleaned deployment implementation. This is the same style of preprocessing and post-processing intended for Raspberry Pi CPU inference.

In [ ]:
import sys
# sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import cv2
from fira_yolo.detector import YOLOv8ONNXDetector

MODEL = PROJECT_ROOT / 'models' / 'yolov8n_fira_416.onnx'
# Replace with a real local image from your dataset when testing.
IMAGE = Path('path/to/test_image.jpg')

if IMAGE.exists():
    frame = cv2.imread(str(IMAGE))
    detector = YOLOv8ONNXDetector(MODEL, image_size=416)
    detections = detector.predict(frame)
    for d in detections:
        print(d)
else:
    print('Set IMAGE to a local test image before running inference.')

## 8. Raspberry Pi benchmark
Do not infer Raspberry Pi performance from a desktop. Measure latency directly on the target Pi 4 with `tools`/`src` in the deployment environment.

In [ ]:
# On the Raspberry Pi, run from the repository root:
# PYTHONPATH=src python -m fira_yolo.benchmark \\
#   --model models/yolov8n_fira_416.onnx \\
#   --image path/to/test_image.jpg \\
#   --warmup 10 \\
#   --iterations 100

## 9. Project outcome
The final deployment artifact is the exported ONNX detector plus a small OpenCV/ONNX Runtime inference layer. The repository keeps the model and inference code versioned while excluding the large competition dataset.